In [1]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")
DATA_DIR = PROJECT_ROOT / "data"

def read_json(file_path: Path | str) -> dict | list:
    data = None
    with open(file_path, "r") as f:
        data = json.load(f)
    return data

def read_jsonl(file_path: Path | str) -> list:
    res = []
    with open(file_path, "r") as f:
        for line in f:
            res.append(json.loads(line))
    return res

def write_json(data: dict | list, file_path: Path | str) -> None:
    with open(file_path, "w") as f:
        json.dump(data, f, indent=4)
        print(f"wrote to {file_path}")
        

Project root: /home/v-homatthew/ctx_editor


In [2]:
tgt_jsonl = Path("/home/v-homatthew/ctx_editor/outputs/2026-03-06/02-22-02/content_filter_errors.jsonl")

In [3]:
tgt_data = read_jsonl(tgt_jsonl)

In [10]:
ex0 = tgt_data[0]
print(list(ex0.keys()))

['timestamp', 'model', 'error_type', 'error_message', 'request_messages', 'error_body']


In [12]:
print(ex0["model"])
ex0["request_messages"]

gpt-5-mini


[{'role': 'system',
  'content': 'As an expert problem solver solve step by step the following mathematical questions.\n\nWhen you have reached a final numerical answer, highlight it on its own line like this: **ANSWER: 42**. Write only the number with no units or extra text.\n'},
 {'role': 'user',
  'content': "how much was charged by Patty's Plumbing for this job?"},
 {'role': 'assistant',
  'content': "I don't have the problem details, so I can't compute a numeric answer yet. Please paste the full problem statement or the numbers involved. To compute the charge I will need (if they apply):\n\n- labor: number of hours and hourly rate (or flat labor charge)\n- materials/parts cost\n- any fixed service or call-out fee\n- any sales tax (rate or amount)\n- any discount or coupon (rate or amount)\n- any other fees (travel, disposal, permit, etc.)\n\nGeneral step-by-step method I will follow once you provide the numbers:\n\n1. Compute labor cost = (hours) × (hourly rate) or use the given f

In [47]:
def convo_to_string(convo: list[dict]) -> str:
    # [{role}]
    # {content}
    # next message, etc.
    lines = []
    for msg in convo:
        content = msg["content"].strip()
        lines.append(f"[[{msg['role']}]]\n{content}")
    return "\n".join(lines)

In [39]:
def get_filter_categories(entry: dict) -> list[str]:
    # entry["error_body"]["innerrror"]["content_filter_result"] is a dict of category to bool
    # return the categories that are True
    res = []
    for cat, val in entry["error_body"]["innererror"]["content_filter_result"].items():
        if val["filtered"]:
            res.append(f"{cat} ({val.get('severity', 'unknown')})")
    return res

In [53]:
len(tgt_data)

2

In [54]:
# general procedure per entry

# print model
# print filter categories
# print convo in readable format

tgt_entry = tgt_data[1]
print(f"Model: {tgt_entry['model']}")
filter_cats = get_filter_categories(tgt_entry)
print(f"Filter categories: {filter_cats}")
convo_str = convo_to_string(tgt_entry["request_messages"])
print(f"---\n{convo_str}")

Model: gpt-5-mini
Filter categories: ['sexual (medium)']
---
[[system]]
As an expert problem solver solve step by step the following mathematical questions.

When you have reached a final numerical answer, highlight it on its own line like this: **ANSWER: 42**. Write only the number with no units or extra text.
[[user]]
how much was charged by Patty's Plumbing for this job?
[[assistant]]
I don't have the problem details, so I can't compute a numeric answer yet. Please paste the full problem statement or the numbers involved. To compute the charge I will need (if they apply):

- labor: number of hours and hourly rate (or flat labor charge)
- materials/parts cost
- any fixed service or call-out fee
- any sales tax (rate or amount)
- any discount or coupon (rate or amount)
- any other fees (travel, disposal, permit, etc.)

General step-by-step method I will follow once you provide the numbers:

1. Compute labor cost = (hours) × (hourly rate) or use the given flat labor charge.
2. Add mate